# Paper Families as Configurable SILVA Architectures

This notebook checks the package mechanisms used to construct DEQ,
MDEQ, Jacobian-regularized, IGNN, implicit-representation, and
diffusion cases. It is a CPU smoke tutorial, not a claim that the
source papers' datasets, schedules, checkpoints, or metrics were run.

All cases share

$$z^\star=f_\theta(z^\star,x).$$

The user selects the state, transition operators, solver, gradient
estimator, dimensions, data, optimizer, and evaluation protocol.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [Path.cwd(), Path("/content/silva-networks"), Path("/content/drive/MyDrive/silva-networks")]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import torch
from torch import nn

from silva_networks import (
    SILVADEQEngine,
    SILVADiffusionEquilibrium,
    SILVAImplicitGraphNetwork,
    SILVAImplicitNeuralRepresentation,
    SILVAMultiscaleClassifier,
    SILVAMultiscaleSegmenter,
    SILVASequenceDEQ,
    SolverConfig,
    jacobian_regularization_loss,
)

torch.manual_seed(12)
fast = SolverConfig(solver="picard", max_iter=3, alpha=0.5)
exact = SolverConfig(
    solver="picard",
    max_iter=3,
    alpha=0.5,
    backward_mode="implicit",
    backward_solver="gmres",
    backward_max_iter=8,
    backward_tol=1e-5,
    backward_stop_mode="relative",
)

## Sequence DEQ

The transition can be relative-attention Transformer or causal
trellis. Memory, local attention, banded adaptive input and output,
weight/projection tying, dropout, and every solver option are
constructor parameters. Custom transition, embedding, and readout
modules are supported as well.

In [ ]:
sequence = SILVASequenceDEQ(
    8,
    vocab_size=24,
    heads=2,
    inner_dim=16,
    memory_length=3,
    local_window=4,
    adaptive_cutoffs=(8, 16),
    adaptive_div_value=2.0,
    embedding_dim=8,
    config=exact,
)
seq = sequence(torch.randint(0, 24, (2, 5)), return_result=True)
targets = torch.randint(0, 24, (2, 5))
sequence.adaptive_loss(seq.state, targets).backward()
print(seq.output.shape, seq.memory.shape, seq.solver_result.solver)
print("normalized probabilities", seq.output.exp().sum(dim=-1))

trellis = SILVASequenceDEQ(
    8,
    input_dim=5,
    output_dim=3,
    mode="trellis",
    tie_embeddings=False,
    config=fast,
)
print("trellis", trellis(torch.randn(2, 5, 5)).shape)

## Multiscale DEQ and Jacobian Regularization

Every resolution is part of one coupled state. Every-to-every
projections and resampling implement cross-scale interaction. The
built-in MDEQ mode uses learned downsampling chains and projected
upsampling; stimulus injection, convolution weight normalization,
and classification/segmentation heads remain selectable.

A paper-selected Jacobian term estimates
$\|J_f(z^\star)\|_F^2$ with Hutchinson probes.

In [ ]:
image = torch.randn(1, 3, 8, 8)
classifier = SILVAMultiscaleClassifier(
    3,
    (4, 6),
    3,
    expansion=1.0,
    groups=2,
    weight_norm=True,
    fusion_mode="mdeq",
    injection_mode="highest",
    config=fast,
)
segmenter = SILVAMultiscaleSegmenter(
    3, (4, 6), 2, expansion=1.0, groups=2, config=fast
)
cls = classifier(image, return_result=True)
print("classification", cls.output.shape, [z.shape for z in cls.states])
print("segmentation", segmenter(image).shape)

matrix = nn.Parameter(0.2 * torch.eye(4))
state = torch.randn(2, 4)
transition = lambda z: torch.tanh(z @ matrix)
penalty = jacobian_regularization_loss(transition, state, samples=2, weight=0.01)
print("Jacobian penalty", float(penalty.detach()))

## IGNN, Implicit Representations, and DEQ-DDIM

These applications change the domain-specific transition while
retaining the same equilibrium and gradient contracts.

In [ ]:
edges = torch.tensor([[0, 1, 2, 3], [1, 2, 3, 0]])
graph = SILVAImplicitGraphNetwork(3, 5, 2, config=fast)
graph.project_recurrent_norm(0.9)
print("IGNN", graph(torch.randn(4, 3), edges).shape)

coordinates = torch.rand(1, 7, 2, requires_grad=True)
inr = SILVAImplicitNeuralRepresentation(
    2, 6, 1, injection="fourier", activation="tanh", config=fast
)
print("INR", inr(coordinates).shape, inr.coordinate_gradient(coordinates).shape)

class ZeroDenoiser(nn.Module):
    def forward(self, x, timestep):
        del timestep
        return torch.zeros_like(x)

ddim = SILVADiffusionEquilibrium(
    ZeroDenoiser(),
    torch.linspace(0.99, 0.5, 10),
    (9, 6, 3, 0, -1),
    config=SolverConfig(solver="picard", max_iter=6, alpha=1.0),
)
diffusion = ddim(torch.randn(1, 1, 3, 3), return_result=True)
print("DDIM", diffusion.output.shape, diffusion.trajectory.shape)

## Beyond the Referenced Cases

`SILVADEQEngine` accepts a user transition over one tensor or a
nested tuple/list state. This is the extension point for a new
architecture; no package family registration is required.

In [ ]:
stimulus = torch.randn(2, 4)
custom_transition = nn.Sequential(nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 4))
engine = SILVADEQEngine(fast)
custom = engine(lambda z: 0.2 * custom_transition(z) + stimulus, torch.zeros_like(stimulus))
print("custom equilibrium", custom.shape)

## Reproduction Boundary

The successful cells establish executable architecture, solver, and
gradient mechanisms. Reproducing a paper's reported numbers still
requires its exact data, preprocessing, training schedule, evaluation,
random-seed policy, and pretrained components.

Cite the SILVA package and the source paper whose architecture or
experimental protocol you instantiate:
https://github.com/jseluis/silva-networks
https://doi.org/10.5281/zenodo.21770099